# Grafo de conhecimento combinado

Um caso de `sample/cases.csv` passa por peças das cinco estratégias de `src/`, cada uma usada onde é mais forte:

| etapa | estratégia | peça |
|---|---|---|
| sentenças, negação, limpeza de rótulo | stopwords | `split_sentences`, `detect_polarity`, `clean_label` (GUARDED_LABEL + NLTK) |
| tokens, medidas, referências de figura | tokenizacao | `ClinicalRegexTokenizer` |
| fronteira dos rótulos | sintagmas | HMM + `chunk_nps` |
| menções, siglas, deduplicação, relações | normalizacao | extratores, `GraphBuilder`, `build_relations` |
| sítios anatômicos e conceitos MeSH | dicionarios | gazetteers + `greedy_match` / `link_label_to_concept` |

O resultado são as tabelas de nós e arestas do contrato comum, gravadas em `data/processed/`.

In [ ]:
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import nltk
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "normalizacao").is_dir())
SRC = ROOT / "src"
# normalizacao e stopwords são pacotes (importados a partir de src/); tokenizacao,
# sintagmas e dicionarios são pastas planas, das quais só importamos módulos com
# nome único (tokenizers, pos_bio, mesh_parser, normalization, matching).
for pasta in (SRC, SRC / "tokenizacao", SRC / "sintagmas", SRC / "dicionarios"):
    if str(pasta) not in sys.path:
        sys.path.append(str(pasta))

from normalizacao.case_reader import read_case
from normalizacao.core import models as nmodels
from normalizacao.core.graph import GraphBuilder
from normalizacao.extractors.common import ExtractedEntity
from normalizacao.extractors.diagnoses import extract_diagnoses
from normalizacao.extractors.exams import ExtractedExamResult, extract_exam_results, extract_exams
from normalizacao.extractors.findings import ExtractedFinding, extract_findings
from normalizacao.extractors.history import extract_history
from normalizacao.extractors.outcomes import extract_outcomes
from normalizacao.extractors.patient import extract_patient
from normalizacao.extractors.relations import ExtractedCaseEntities, _add_unique_edge, build_relations
from normalizacao.extractors.symptoms import extract_symptoms
from normalizacao.extractors.treatments import extract_medications, extract_treatments
from normalizacao.normalizers.case import CaseNormalizer
from stopwords.extractors.common import detect_polarity, split_sentences
from stopwords.lexicon.label_cleaning import clean_label
from stopwords.lexicon.loader import load_list

import pos_bio
from matching import greedy_match, link_label_to_concept, match_text
from mesh_parser import load_gazetteer_rows, rows_to_raw_gazetteer
from normalization import build_normalized_gazetteer, normalize_token
from tokenizers import ClinicalRegexTokenizer

nltk.download("punkt_tab", quiet=True)
# O esquema (docs/02-esquema-grafo.md) inclui SAME_AS, que a lista de normalizacao não tem.
nmodels.ALLOWED_RELATIONS.add("SAME_AS")


def mostrar(df, n=15, largura=70):
    """Imprime as n primeiras linhas; o restante vira um contador."""
    print(df.head(n).to_string(index=False, max_colwidth=largura))
    if len(df) > n:
        print(f"… (+{len(df) - n} linhas)")


def curto(texto, n=90):
    texto = " ".join(str(texto).split())
    return texto if len(texto) <= n else texto[: n - 1] + "…"


print("raiz do projeto:", ROOT)

raiz do projeto: /Users/caiomellonidossantos/codes/PNL-CGGJL/project1


## 1. Caso
Lido de `sample/cases.csv` com o leitor de normalizacao (valida colunas e converte a idade para `Decimal`).

In [2]:
CASE_ID = "PMC5137649_01"

caso = read_case(ROOT / "sample" / "cases.csv", CASE_ID)
texto = caso.case_text

print(f"case_id={caso.case_id}  article_id={caso.article_id}  age={caso.age}  gender={caso.gender}")
print(f"{len(texto)} caracteres; início do texto:\n")
print(texto[:700] + ("…" if len(texto) > 700 else ""))

case_id=PMC5137649_01  article_id=PMC5137649  age=44.0  gender=Female
2406 caracteres; início do texto:

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence…


## 2. Sentenças — *stopwords*
`split_sentences` só corta em `.` que não precede dígito (`12,476.5` fica inteiro) e preserva os offsets.
As sentenças dão o escopo da negação e das ligações `LOCATED_IN`.

In [3]:
sentencas = [s for s in split_sentences(texto) if s.text.strip()]


def sentenca_de(posicao):
    return max((s for s in sentencas if s.start <= posicao), key=lambda s: s.start, default=sentencas[0])


df_sentencas = pd.DataFrame(
    [{"idx": s.index, "start": s.start, "end": s.end, "sentença": curto(s.text, 100)} for s in sentencas]
)
print(f"{len(sentencas)} sentenças")
mostrar(df_sentencas, n=10, largura=100)

18 sentenças
 idx  start  end                                                                                             sentença
   0      0  139 A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain…
   1    140  266 Her past medical, family and medication history were otherwise non-contributory and her physical ex…
   2    267  412 She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the …
   3    413  618 She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuri…
   4    619  910 FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular m…
   5    911  969                                            The patient was therefore referred for surgical resection
   7    974 1022                                                     A laparoscopic distal pancreatectomy was planned
   8   1023 1245 At the time of the laparos

## 3. Tokens — *tokenizacao*
Tokenizador de regex clínico: número e unidade coladas saem separados (`12,476.5` + `ng/ml`) e `(Fig 1)` vira um token só.
Os tokens alimentam o POS tagger, as medidas dos achados e a rejeição de doses falsas.

In [4]:
tokens = ClinicalRegexTokenizer().tokenize(texto)
assert all(texto[t.start:t.end] == t.text for t in tokens)

print(f"{len(tokens)} tokens por tipo:", dict(Counter(t.kind for t in tokens).most_common()))
df_tokens = pd.DataFrame([{"index": t.index, "text": t.text, "kind": t.kind, "start": t.start, "end": t.end} for t in tokens])
print("\nTokens que não são palavra nem pontuação:")
mostrar(df_tokens[~df_tokens.kind.isin(["WORD", "PUNCTUATION"])], n=20)

414 tokens por tipo: {'WORD': 358, 'PUNCTUATION': 33, 'NUMBER': 9, 'UNIT': 8, 'FIGURE_REF': 3, 'NUM_HYPHEN_WORD': 2, 'NUMBER_RANGE': 1}

Tokens que não são palavra nem pontuação:
 index        text            kind  start  end
     1 44-year-old NUM_HYPHEN_WORD      2   13
     6       3-day NUM_HYPHEN_WORD     37   42
    49           6          NUMBER    337  338
    50          cm            UNIT    338  340
    61     (Fig 1)      FIGURE_REF    405  412
    77           6          NUMBER    516  517
    78          cm            UNIT    517  519
    80           9          NUMBER    522  523
    81          cm            UNIT    523  525
    91     (Fig 2)      FIGURE_REF    584  591
   125    12,476.5          NUMBER    777  785
   126       ng/ml            UNIT    785  790
   134        19-9    NUMBER_RANGE    823  827
   137           6          NUMBER    837  838
   138       iu/ml            UNIT    838  843
   349         9.5          NUMBER   2070 2073
   350          cm    

## 4. POS e sintagmas nominais — *sintagmas*
Os tokens acima viram `pos_bio.Token` (unidade marcada com `role=UNIT`, referência de figura forçada a pontuação).
O HMM etiqueta, as regras de reparo corrigem e `chunk_nps` devolve os sintagmas `DET? (NUM UNIT?)? MOD* NOUN+`,
que definem onde começa e termina cada rótulo.

In [5]:
tagger = pos_bio.HMMTagger()
ptokens = []
for t in tokens:
    pt = pos_bio.Token(t.text, t.start, t.end, t.index)
    if t.kind == "UNIT":
        pt.role = "UNIT"
    ptokens.append(pt)

restricoes = pos_bio.build_constraints(ptokens, tagger, pos_bio.HEAD_NOUNS)
for i, t in enumerate(tokens):
    if t.kind == "FIGURE_REF":
        restricoes[i] = {"."}
tagger.tag(ptokens, restricoes)
reparos = pos_bio.repair_tags(ptokens)
chunks = pos_bio.chunk_nps(ptokens)
tipo_lexico = {(sp.i0, sp.i1): sp.type for sp in pos_bio.find_spans(chunks) if sp.layer == 0}
np_por_token = {t.index: ch for ch in chunks for t in ch}

print("POS da primeira sentença:")
print(" ".join(f"{t.text}/{t.tag}" for t in ptokens if t.start < sentencas[0].end))
print(f"\nEtiquetas: {dict(Counter(t.tag for t in ptokens).most_common())}")
print(f"Reparos aplicados: {[(r[0], r[1]) for r in reparos]}")

df_nps = pd.DataFrame(
    [
        {
            "sintagma": pos_bio.surface(ch),
            "start": ch[0].start,
            "end": ch[-1].end,
            "tipo_léxico": tipo_lexico.get((ch[0].index, ch[-1].index + 1), ""),
        }
        for ch in chunks
    ]
)
print(f"\n{len(chunks)} sintagmas nominais, {int((df_nps['tipo_léxico'] != '').sum())} com tipo no léxico:")
mostrar(df_nps, n=20)

verbos = Counter(t.text.lower() for t in ptokens if t.tag == "VERB" and t.index not in np_por_token)
print("\nVerbos fora de sintagma (candidatos a gatilho):", dict(verbos.most_common(12)))

POS da primeira sentença:
A/DET 44-year-old/ADJ woman/NOUN presented/VERB with/ADP a/DET 3-day/ADJ history/NOUN of/ADP right/NOUN flank/NOUN and/CONJ lower/ADJ quadrant/NOUN abdominal/ADJ pain/NOUN associated/VERB with/ADP nausea/NOUN and/CONJ constipation/NOUN

Etiquetas: {'NOUN': 111, 'VERB': 58, 'DET': 57, 'ADJ': 46, 'ADP': 41, '.': 36, 'CONJ': 23, 'PRON': 9, 'NUM': 9, 'X': 9, 'ADV': 8, 'PRT': 7}
Reparos aplicados: [('R-relverb', 'revealed'), ('R-passiva', 'discharged'), ('R-premod', 'contrast')]

95 sintagmas nominais, 49 com tipo no léxico:
                             sintagma  start  end    tipo_léxico
                  A 44-year-old woman      0   19               
                      a 3-day history     35   50               
                          right flank     54   65 AnatomicalSite
        lower quadrant abdominal pain     70   99        Symptom
                               nausea    116  122        Symptom
                         constipation    127  139        S

## 5. Siglas do caso — *normalizacao*
Definições `termo por extenso (SIGLA)` valem só dentro deste caso; o `GraphBuilder` as expande ao normalizar os rótulos.

In [6]:
siglas = CaseNormalizer(texto).acronym_definitions
print(f"{len(siglas)} siglas definidas no texto")
mostrar(pd.DataFrame([{"sigla": k, "expansão": v} for k, v in siglas.items()]))

2 siglas definidas no texto
sigla                 expansão
  CEA carcinoembryonic antigen
   CA     carbohydrate antigen


## 6. Menções brutas — *normalizacao*
Os extratores de normalizacao rodam num grafo rascunho, na mesma ordem de `normalizacao/pipeline.py`.
O rascunho guarda o rótulo bruto de cada nó para que a menção possa ser localizada no texto depois.

In [7]:
class GrafoRascunho(GraphBuilder):
    """GraphBuilder que lembra os rótulos brutos recebidos por cada nó."""

    def __init__(self, case_id, case_text):
        super().__init__(case_id, case_text)
        self.rotulos_brutos = defaultdict(list)

    def add_node(self, node_type, raw_label, attributes=None, *, deduplicate=True):
        node = super().add_node(node_type, raw_label, attributes, deduplicate=deduplicate)
        self.rotulos_brutos[node.node_id].append(raw_label)
        return node


rascunho = GrafoRascunho(caso.case_id, texto)
brutas = ExtractedCaseEntities(
    patient=extract_patient(caso, rascunho),
    symptoms=tuple(extract_symptoms(texto, rascunho)),
    histories=tuple(extract_history(texto, rascunho)),
    exams=tuple(extract_exams(texto, rascunho)),
    exam_results=tuple(extract_exam_results(texto, rascunho)),
    findings=tuple(extract_findings(texto, rascunho)),
    diagnoses=tuple(extract_diagnoses(texto, rascunho)),
    medications=tuple(extract_medications(texto, rascunho)),
    treatments=tuple(extract_treatments(texto, rascunho)),
    outcomes=tuple(extract_outcomes(texto, rascunho)),
)

CAMPOS = ("symptoms", "histories", "exams", "findings", "diagnoses", "medications", "treatments", "outcomes")


def entidade(item):
    return item.entity if isinstance(item, ExtractedFinding) else item


linhas = []
for campo in CAMPOS:
    for item in getattr(brutas, campo):
        ent = entidade(item)
        linhas.append({"node_id": ent.node.node_id, "type": ent.node.type, "label": ent.node.label,
                       "attributes": nmodels.serialize_attributes(ent.node.attributes), "trigger": ent.trigger,
                       "char_start": ent.char_start, "char_end": ent.char_end})
for r in brutas.exam_results:
    linhas.append({"node_id": r.node.node_id, "type": "ExamResult", "label": r.node.label,
                   "attributes": nmodels.serialize_attributes(r.node.attributes), "trigger": f"exame {r.exam_node.label}",
                   "char_start": r.char_start, "char_end": r.char_end})
df_brutas = pd.DataFrame(linhas)

print(f"{len(df_brutas)} menções, {len(rascunho.nodes)} nós no rascunho")
print(df_brutas["type"].value_counts().to_dict())
mostrar(df_brutas[["node_id", "type", "label", "attributes", "trigger"]], n=40, largura=55)

28 menções, 32 nós no rascunho
{'Finding': 10, 'Exam': 5, 'Symptom': 3, 'Outcome': 3, 'Diagnosis': 2, 'Treatment': 2, 'ExamResult': 2, 'Medication': 1}
node_id       type                                         label                                              attributes                            trigger
     S1    Symptom right flank and lower quadrant abdominal pain                       polarity=present; duration=3 days                     presented with
     S2    Symptom                                        nausea                                        polarity=present                     presented with
     S3    Symptom                                  constipation                                        polarity=present                     presented with
     E1       Exam                           computed tomography                                        modality=imaging                computed tomography
     E2       Exam                         endoscopic ultrasound    

## 7. Refinamento das menções
Cada menção é localizada no texto e passa por quatro correções, cada uma vinda de uma estratégia:

1. **tokenizacao** — corta o rótulo antes de um token `FIGURE_REF` e o nome de exame laboratorial na pontuação
   (`0-0.04 ng/ml), creatine kinase` → `creatine kinase`); recalcula o tamanho do achado com a sequência de tokens
   `N UNIT (x N UNIT)*`; descarta `Medication` cuja "dose" é parte de um token de concentração (`6iu/ml`).
2. **sintagmas** — o rótulo de `Finding` vira o sintagma nominal do núcleo; em `Symptom`/`History`/`Diagnosis`,
   só quando o rótulo começa fora de um sintagma (`and bleeding`, `performed and no …`). Artigo, número e unidade saem.
3. **stopwords** — negação detectada na oração que antecede a menção (só `present → absent`).
4. **stopwords** — `clean_label` com a lista do NLTK e a lista de proteção, aplicado só ao rótulo (GUARDED_LABEL).

In [8]:
LIVRES = {"Symptom", "History", "Diagnosis", "Finding"}
COM_POLARIDADE = {"Symptom", "Finding", "History", "Outcome"}
STOP_NLTK = load_list("nltk_stopwords")
PROTEGIDAS = load_list("protection_list")
figuras = [t for t in tokens if t.kind == "FIGURE_REF"]
CORTE_ORACAO = re.compile(r"[,;]|\bbut\b", re.I)


def localizar(inicio, fim, candidatos, nucleo=False):
    """Span da menção dentro da evidência [inicio, fim).

    Procura cada candidato literalmente; com `nucleo`, recorre à última palavra do rótulo
    (`and bleeding` não aparece contíguo em "no evidence of bleeding").
    """
    trecho = texto[inicio:fim].lower()
    for candidato in candidatos:
        alvo = candidato.strip().lower()
        if alvo and (i := trecho.find(alvo)) >= 0:
            return inicio + i, inicio + i + len(alvo)
    if nucleo:
        for candidato in candidatos:
            palavras = re.findall(r"[A-Za-z][\w-]*", candidato)
            if palavras and (m := re.search(rf"\b{re.escape(palavras[-1])}\b", texto[inicio:fim], re.I)):
                return inicio + m.start(), inicio + m.end()
    return None


def aparar(inicio, fim):
    trecho = texto[inicio:fim]
    esquerda = len(trecho) - len(trecho.lstrip(" \t\n,;:("))
    direita = len(trecho.rstrip(" \t\n,;:("))
    return inicio + esquerda, inicio + direita


def ptokens_em(inicio, fim):
    return [t for t in ptokens if inicio <= t.start and t.end <= fim and t.text[:1].isalnum()]


def miolo_do_np(chunk):
    miolo = [t for t in chunk if t.tag not in ("DET", "NUM") and t.role != "UNIT"]
    return (miolo[0].start, miolo[-1].end) if miolo else None


def medidas(inicio, fim):
    """Dimensões `N UNIT (x N UNIT)*` formadas por tokens dentro de [inicio, fim)."""
    seq = [t for t in tokens if inicio <= t.start and t.end <= fim]
    achadas, i = [], 0
    while i < len(seq) - 1:
        if seq[i].kind == "NUMBER" and seq[i + 1].kind == "UNIT":
            valores, unidade, j = [seq[i].text], seq[i + 1].text, i + 2
            while j + 2 < len(seq) and seq[j].text.lower() == "x" and seq[j + 1].kind == "NUMBER" and seq[j + 2].kind == "UNIT":
                valores.append(seq[j + 1].text)
                unidade = seq[j + 2].text
                j += 3
            achadas.append((seq[i].start, seq[j - 1].end, " x ".join(valores) + " " + unidade.lower()))
            i = j
        else:
            i += 1
    return achadas


def juntar(toks):
    """Texto dos tokens, preservando o espaçamento original entre vizinhos."""
    partes = [toks[0].text]
    for anterior, atual in zip(toks, toks[1:]):
        partes.append(texto[anterior.end:atual.start] if atual.index == anterior.index + 1 else " ")
        partes.append(atual.text)
    return "".join(partes)


def nome_do_analito(toks):
    """Nome do exame dentro dos tokens: sem parentéticos, cortado em `)`/`,` soltos e em `(` sem fechamento."""
    segmentos, atual, abertos = [], [], []
    for t in toks:
        if t.text == "(":
            abertos.append(len(atual))
            atual.append(t)
        elif t.text == ")" and abertos:
            del atual[abertos.pop():]
        elif t.text in {")", ","}:
            segmentos.append(atual)
            atual, abertos = [], []
        else:
            atual.append(t)
    if abertos:
        atual = atual[:abertos[0]]
    segmentos.append(atual)
    conteudo = [s for s in segmentos if any(t.kind == "WORD" and t.text.lower() not in STOP_NLTK for t in s)]
    return conteudo[-1] if conteudo else []


def janela_da_oracao(span):
    sentenca = sentenca_de(span[0])
    cortes = list(CORTE_ORACAO.finditer(texto, sentenca.start, span[0]))
    inicio = cortes[-1].end() if cortes else sentenca.start
    return texto[inicio:span[1]]


def refinar(node, inicio, fim, trigger):
    tipo, attrs, notas = node.type, dict(node.attributes), []
    rotulo = node.label
    brutos = rascunho.rotulos_brutos[node.node_id]
    if tipo == "Medication":
        candidatos = (trigger,)
    elif tipo in LIVRES:
        candidatos = (*brutos, node.label)
    else:
        candidatos = (*brutos, node.label, trigger)
    span = localizar(inicio, fim, candidatos, nucleo=tipo in LIVRES)

    if tipo == "Medication" and span is not None:
        pos_unidade = span[1] - len(str(attrs["dose_unit"]))
        token_unidade = next((t for t in tokens if t.start <= pos_unidade < t.end), None)
        if token_unidade is not None and token_unidade.text.lower() != str(attrs["dose_unit"]).lower():
            notas.append(("tokenizacao", f"descartada: '{attrs['dose_unit']}' faz parte do token de unidade '{token_unidade.text}'"))
            return {"tipo": tipo, "rotulo": rotulo, "attrs": attrs, "span": span, "notas": notas, "descartar": True}

    if tipo == "Exam" and attrs.get("modality") == "laboratory" and span is not None:
        mantidos = nome_do_analito([t for t in tokens if span[0] <= t.start and t.end <= span[1]])
        if mantidos and juntar(mantidos) != texto[span[0]:span[1]]:
            notas.append(("tokenizacao", f"corta na pontuação: '{texto[span[0]:span[1]]}'"))
            span = (mantidos[0].start, mantidos[-1].end)
            rotulo = juntar(mantidos)

    if tipo in LIVRES and span is not None:
        for figura in figuras:
            if span[0] < figura.start < span[1]:
                span = aparar(span[0], figura.start)
                notas.append(("tokenizacao", f"remove {figura.text!r}"))

        palavras = ptokens_em(*span)
        chunk = None
        if tipo == "Finding" and palavras:
            chunk = np_por_token.get(palavras[-1].index)
        elif palavras and palavras[0].index not in np_por_token:
            chunk = next((np_por_token[t.index] for t in palavras if t.index in np_por_token), None)
        novo = miolo_do_np(chunk) if chunk else None
        if novo and novo != span and (tipo != "Finding" or novo[0] <= palavras[-1].start < novo[1]):
            span = novo
            notas.append(("sintagmas", f"sintagma '{pos_bio.surface(chunk)}'"))

        if tipo == "Finding":
            sentenca = sentenca_de(span[0])
            inicio_np = chunk[0].start if chunk else span[0]
            tamanho = next(
                (
                    m[2]
                    for m in reversed(medidas(sentenca.start, sentenca.end))
                    if inicio_np < m[1] <= span[1] or re.fullmatch(r"\s*measuring\s*", texto[span[1]:m[0]], re.I)
                ),
                None,
            )
            if tamanho != attrs.get("size"):
                notas.append(("tokenizacao", f"size {attrs.get('size')!r} → {tamanho!r}"))
                attrs["size"] = tamanho

        rotulo = texto[span[0]:span[1]]

    if tipo in COM_POLARIDADE and span is not None and attrs.get("polarity") == "present":
        if detect_polarity(janela_da_oracao(span)) == "absent":
            attrs["polarity"] = "absent"
            notas.append(("stopwords", f"negação em '{curto(janela_da_oracao(span), 60)}'"))

    limpo = clean_label(rotulo, STOP_NLTK, PROTEGIDAS)
    if limpo != rotulo:
        notas.append(("stopwords", f"clean_label '{rotulo}' → '{limpo}'"))
        rotulo = limpo

    return {"tipo": tipo, "rotulo": rotulo, "attrs": attrs, "span": span, "notas": notas, "descartar": False}


registros = []
for campo in CAMPOS:
    for item in getattr(brutas, campo):
        ent = entidade(item)
        registros.append({"campo": campo, "bruta": ent, **refinar(ent.node, ent.char_start, ent.char_end, ent.trigger)})
exames_de_resultado = []
for r in brutas.exam_results:
    exames_de_resultado.append((r, refinar(r.exam_node, r.char_start, r.char_end, "")))

normalizador = CaseNormalizer(texto)
linhas = [
    {"node_id": reg["bruta"].node.node_id, "type": reg["tipo"], "antes": reg["bruta"].node.label,
     "depois": "—" if reg["descartar"] else normalizador.normalize_entity_label(reg["rotulo"]), "estratégia": est, "correção": nota}
    for reg in registros + [{"bruta": ExtractedEntity(r.exam_node, "", "", 0, 0), **ref} for r, ref in exames_de_resultado]
    for est, nota in reg["notas"]
]
df_refino = pd.DataFrame(linhas).drop_duplicates()
print(f"{len(df_refino)} correções; por estratégia: {df_refino['estratégia'].value_counts().to_dict()}")
mostrar(df_refino, n=40, largura=60)

18 correções; por estratégia: {'tokenizacao': 7, 'sintagmas': 6, 'stopwords': 5}
node_id       type                                         antes                                    depois  estratégia                                                     correção
     S1    Symptom right flank and lower quadrant abdominal pain right flank lower quadrant abdominal pain   stopwords clean_label 'right flank and lower quadrant abdominal pai...
     F1    Finding             pancreatic echotexture and a cyst                                      cyst   sintagmas                                            sintagma 'a cyst'
     F1    Finding             pancreatic echotexture and a cyst                                      cyst tokenizacao                                       size None → '6 x 9 cm'
     F2    Finding                               fna of the cyst                                      cyst   sintagmas                                          sintagma 'the cyst'
     F5    Finding 

## 8. Grafo normalizado e relações — *normalizacao*
As menções refinadas entram num `GraphBuilder` novo, que expande siglas, normaliza o rótulo e deduplica por
atributos de identidade (sintoma presente e ausente continuam nós distintos). A evidência de cada menção é
aparada para que `case_text[char_start:char_end] == evidence_text`, e `build_relations` cria as arestas.

In [9]:
def ancorar(inicio, fim):
    inicio, fim = aparar(inicio, fim)
    return texto[inicio:fim], inicio, fim


final = GraphBuilder(caso.case_id, texto)
paciente = extract_patient(caso, final)
por_campo = defaultdict(list)
for reg in registros:
    if reg["descartar"]:
        continue
    bruta = reg["bruta"]
    # Achados com tamanhos diferentes são observações distintas (ex.: cisto de 6 x 9 cm na EUS e de 9.5 x 4.5 x 2.0 cm na patologia).
    rotulo_normalizado = final.normalizer.normalize_entity_label(reg["rotulo"])
    tamanho = reg["attrs"].get("size")
    conflito_de_tamanho = tamanho is not None and any(
        n.type == reg["tipo"] and n.label == rotulo_normalizado and n.attributes.get("size") not in (None, tamanho)
        for n in final.nodes
    )
    node = final.add_node(reg["tipo"], reg["rotulo"], reg["attrs"], deduplicate=not conflito_de_tamanho)
    evidencia, inicio, fim = ancorar(bruta.char_start, bruta.char_end)
    reg["final"] = ExtractedEntity(node, evidencia, bruta.trigger, inicio, fim)
    por_campo[reg["campo"]].append(reg["final"])

resultados = []
for r, ref in exames_de_resultado:
    exame = final.add_node("Exam", ref["rotulo"], ref["attrs"])
    resultado = final.add_node("ExamResult", rascunho.rotulos_brutos[r.node.node_id][0], dict(r.node.attributes), deduplicate=False)
    evidencia, inicio, fim = ancorar(r.char_start, r.char_end)
    resultados.append(ExtractedExamResult(resultado, exame, evidencia, inicio, fim))

entidades = ExtractedCaseEntities(
    patient=paciente,
    symptoms=tuple(por_campo["symptoms"]),
    histories=tuple(por_campo["histories"]),
    exams=tuple(por_campo["exams"]),
    exam_results=tuple(resultados),
    findings=tuple(ExtractedFinding(ent, ()) for ent in por_campo["findings"]),
    diagnoses=tuple(por_campo["diagnoses"]),
    medications=tuple(por_campo["medications"]),
    treatments=tuple(por_campo["treatments"]),
    outcomes=tuple(por_campo["outcomes"]),
)
build_relations(final, entidades)

impacto = final.normalization_impact()
print(f"{len(final.nodes)} nós e {len(final.edges)} arestas")
print("impacto da normalização:", impacto.to_dict())
print("nós por tipo:", dict(Counter(n.type for n in final.nodes)))
print("arestas por relação:", dict(Counter(e.relation for e in final.edges)))

28 nós e 30 arestas
impacto da normalização: {'mentions_processed': 30, 'raw_nodes': 28, 'normalized_nodes': 28, 'labels_changed': 3, 'nodes_merged': 0}
nós por tipo: {'Patient': 1, 'Symptom': 3, 'Exam': 6, 'Finding': 9, 'Diagnosis': 2, 'Treatment': 2, 'Outcome': 3, 'ExamResult': 2}
arestas por relação: {'HAS_SYMPTOM': 3, 'DIAGNOSED_WITH': 2, 'HAS_OUTCOME': 3, 'HAS_RESULT': 2, 'UNDERWENT_EXAM': 6, 'REVEALS': 3, 'HAS_FINDING': 6, 'TREATED_WITH': 2, 'SUPPORTS': 2, 'REVISES': 1}


## 9. Sítios anatômicos — *dicionarios*
O gazetteer anatômico curado (`anatomical_site_gazetteer.csv`) é casado contra os tokens do caso.
Como em dicionarios, o órgão embutido num diagnóstico ou achado (`gastric duplication cyst`) não vira sítio.
Cada sítio se liga por `LOCATED_IN` ao `Symptom`/`Finding`/`Treatment` mais próximo na mesma sentença.

In [10]:
gaz_anatomia = build_normalized_gazetteer(
    rows_to_raw_gazetteer(load_gazetteer_rows(SRC / "dicionarios" / "gazetteer" / "anatomical_site_gazetteer.csv"), "anatomical_site")
)
EMBUTE_ORGAO = {"Diagnosis", "Finding"}
ANCORAS = {"Symptom", "Finding", "Treatment"}
LATERALIDADE = re.compile(r"\b(left|right|bilateral)\b", re.I)
REGIAO = re.compile(r"\b(upper|lower|proximal|distal|anterior|posterior|body/tail|tail)\b", re.I)

ocupados = [reg["span"] for reg in registros if not reg["descartar"] and reg["tipo"] in EMBUTE_ORGAO and reg["span"]]
ancoras = [reg for reg in registros if not reg["descartar"] and reg["tipo"] in ANCORAS and reg["span"]]

linhas = []
for m in match_text([t.text for t in tokens], gaz_anatomia):
    inicio, fim = tokens[m.start].start, tokens[m.end - 1].end
    superficie = texto[inicio:fim]
    if any(s[0] <= inicio and fim <= s[1] for s in ocupados):
        linhas.append({"sítio": superficie, "código": m.code, "estratégia": m.strategy, "resultado": "descartado: dentro de diagnóstico/achado"})
        continue
    sentenca = sentenca_de(inicio)
    candidatas = [reg for reg in ancoras if sentenca.start <= reg["span"][0] < sentenca.end]
    if not candidatas:
        linhas.append({"sítio": superficie, "código": m.code, "estratégia": m.strategy, "resultado": "descartado: sem âncora na sentença"})
        continue
    ancora = min(candidatas, key=lambda reg: min(abs(reg["span"][0] - fim), abs(inicio - reg["span"][1])))
    vizinhanca = texto[max(sentenca.start, inicio - 30):fim + 30]
    lateral, regiao = LATERALIDADE.search(vizinhanca), REGIAO.search(vizinhanca)
    sitio = final.add_node(
        "AnatomicalSite",
        m.preferred_term,
        {"laterality": lateral.group(1).lower() if lateral else None, "region_qualifier": regiao.group(1).lower() if regiao else None},
    )
    evidencia, s_ini, s_fim = ancorar(sentenca.start, sentenca.end)
    _add_unique_edge(final, ancora["final"].node, sitio, "LOCATED_IN",
                     {"evidence_text": evidencia, "trigger": superficie, "certainty": "asserted", "char_start": s_ini, "char_end": s_fim})
    conceito = final.add_node("Concept", m.preferred_term, {"vocabulary": "local", "code": m.code, "preferred_term": m.preferred_term})
    _add_unique_edge(final, sitio, conceito, "SAME_AS", {"strategy": m.strategy, "score": m.score})
    linhas.append({"sítio": superficie, "código": m.code, "estratégia": m.strategy,
                   "resultado": f"{sitio.node_id} ← LOCATED_IN ← {ancora['final'].node.node_id} ({ancora['final'].node.label})"})

print(f"{len(linhas)} ocorrências de termos anatômicos")
mostrar(pd.DataFrame(linhas), n=30, largura=70)

16 ocorrências de termos anatômicos
     sítio  código estratégia                                                        resultado
 abdominal ANAT033      exact A1 ← LOCATED_IN ← S1 (right flank lower quadrant abdominal pain)
   stomach ANAT001      exact                               descartado: sem âncora na sentença
  pancreas ANAT002      exact                               descartado: sem âncora na sentença
pancreatic ANAT002      exact                                      A2 ← LOCATED_IN ← F1 (cyst)
   stomach ANAT001      exact                                      A3 ← LOCATED_IN ← F1 (cyst)
pancreatic ANAT002      exact                         descartado: dentro de diagnóstico/achado
   stomach ANAT001      exact                                    A3 ← LOCATED_IN ← F3 (lesion)
   stomach ANAT001      exact                                    A3 ← LOCATED_IN ← F3 (lesion)
  pancreas ANAT002      exact                                    A2 ← LOCATED_IN ← F3 (lesion)
  pancreas ANA

## 10. Conceitos MeSH — *dicionarios*
O gazetteer MeSH é carregado uma vez e cada tipo consulta só a sua categoria (mesma rota de `dicionarios/pipeline.py`).
Como os rótulos já estão curtos, um casamento parcial só é aceito se cobrir o último token (o núcleo):
`right flank lower quadrant abdominal pain` liga a *Abdominal Pain*, mas um rótulo não liga a um termo solto do meio.
O casamento aproximado (rapidfuzz, limiar 85, mínimo de 5 caracteres) só roda quando nada casa exatamente.

In [11]:
inicio_carga = time.time()
linhas_mesh = load_gazetteer_rows(SRC / "dicionarios" / "gazetteer" / "mesh_gazetteer.csv")
gaz_mesh = {
    categoria: build_normalized_gazetteer(rows_to_raw_gazetteer(linhas_mesh, categoria))
    for categoria in ("diseases", "mental_disorders", "drugs", "exams", "treatments")
}
print(f"gazetteer MeSH: {len(linhas_mesh)} termos em {time.time() - inicio_carga:.1f}s;",
      {categoria: len(chaves) for categoria, chaves in gaz_mesh.items()})

ROTA_MESH = {
    "Symptom": ("diseases",),
    "Finding": ("diseases",),
    "Diagnosis": ("diseases", "mental_disorders"),
    "Exam": ("exams",),
    "Treatment": ("treatments",),
    "Medication": ("drugs",),
}


def ligar(rotulo, gazetteer):
    toks = nltk.word_tokenize(rotulo)
    validos = [i for i, tok in enumerate(toks) if normalize_token(tok)]
    if not validos:
        return None
    casamentos, _ = greedy_match(toks, gazetteer)
    if casamentos:
        no_nucleo = [m for m in casamentos if m.end == validos[-1] + 1]
        return max(no_nucleo, key=lambda m: m.end - m.start) if no_nucleo else None
    return link_label_to_concept(rotulo, gazetteer)


linhas = []
for node in [n for n in final.nodes if n.type in ROTA_MESH]:
    for categoria in ROTA_MESH[node.type]:
        m = ligar(node.label, gaz_mesh[categoria])
        if m is None:
            continue
        conceito = final.add_node("Concept", m.preferred_term, {"vocabulary": "MeSH", "code": m.code, "preferred_term": m.preferred_term})
        _add_unique_edge(final, node, conceito, "SAME_AS", {"strategy": m.strategy, "score": m.score})
        linhas.append({"node_id": node.node_id, "type": node.type, "label": node.label, "categoria": categoria,
                       "code": m.code, "preferred_term": m.preferred_term, "estratégia": m.strategy,
                       "score": None if m.score is None else round(m.score, 1)})
        break
    else:
        linhas.append({"node_id": node.node_id, "type": node.type, "label": node.label, "categoria": "—",
                       "code": "", "preferred_term": "", "estratégia": "sem casamento", "score": None})

df_mesh = pd.DataFrame(linhas)
ligados = int((df_mesh["code"] != "").sum())
print(f"{ligados} de {len(df_mesh)} nós ligáveis ganharam SAME_AS")
mostrar(df_mesh, n=40, largura=45)

gazetteer MeSH: 174006 termos em 2.9s; {'diseases': 58062, 'mental_disorders': 3058, 'drugs': 93264, 'exams': 7762, 'treatments': 10838}
13 de 22 nós ligáveis ganharam SAME_AS
node_id      type                                     label  categoria    code                          preferred_term    estratégia score
     S1   Symptom right flank lower quadrant abdominal pain   diseases D015746                          Abdominal Pain       longest  None
     S2   Symptom                                    nausea   diseases D009325                                  Nausea         exact  None
     S3   Symptom                              constipation   diseases D003248                            Constipation         exact  None
     E1      Exam                       computed tomography      exams D014054                              Tomography         exact  None
     E2      Exam                     endoscopic ultrasound          —                                                 sem casame

## 11. Tabelas finais
Nós e arestas no contrato comum. O rótulo de `ExamResult` é refeito a partir de `value` e `unit`, para usar a unidade canônica
(`ng/mL`) em vez da forma minúscula que a normalização de texto produz. As verificações conferem os offsets de toda evidência e o
domínio de cada relação conforme `docs/01-dados-a-extrair.md` §2 e `docs/02-esquema-grafo.md`.

In [12]:
def rotulo_final(node):
    if node.type == "ExamResult" and node.attributes.get("unit"):
        return f"{format(node.attributes['value'], 'f')} {node.attributes['unit']}"
    return node.label


nos = pd.DataFrame([{**n.to_row(), "label": rotulo_final(n)} for n in final.nodes])[["case_id", "node_id", "type", "label", "attributes"]]
arestas = pd.DataFrame(final.edge_rows())[["case_id", "edge_id", "source_id", "target_id", "relation", "attributes"]]

DOMINIO = {
    "HAS_SYMPTOM": ({"Patient"}, {"Symptom"}),
    "HAS_HISTORY": ({"Patient"}, {"History"}),
    "UNDERWENT_EXAM": ({"Patient"}, {"Exam"}),
    "HAS_RESULT": ({"Exam"}, {"ExamResult"}),
    "REVEALS": ({"Exam", "Treatment"}, {"Finding"}),
    "HAS_FINDING": ({"Patient"}, {"Finding"}),
    "SUPPORTS": ({"Symptom", "Finding", "ExamResult", "History"}, {"Diagnosis"}),
    "DIAGNOSED_WITH": ({"Patient"}, {"Diagnosis"}),
    "TREATED_WITH": ({"Patient", "Diagnosis"}, {"Treatment", "Medication"}),
    "LOCATED_IN": ({"Symptom", "Finding", "Treatment"}, {"AnatomicalSite"}),
    "HAS_OUTCOME": ({"Patient"}, {"Outcome"}),
    "REVISES": ({"Diagnosis"}, {"Diagnosis"}),
    "SAME_AS": ({"Symptom", "Finding", "Exam", "Diagnosis", "Medication", "Treatment", "AnatomicalSite"}, {"Concept"}),
}
tipo_de = {n.node_id: n.type for n in final.nodes}
fora_do_dominio = [e.edge_id for e in final.edges
                   if tipo_de[e.source_id] not in DOMINIO[e.relation][0] or tipo_de[e.target_id] not in DOMINIO[e.relation][1]]
ancoradas = [e for e in final.edges if "char_start" in e.attributes]
desalinhadas = [e.edge_id for e in ancoradas
                if texto[e.attributes["char_start"]:e.attributes["char_end"]] != e.attributes["evidence_text"]]
print(f"offsets: {len(ancoradas) - len(desalinhadas)}/{len(ancoradas)} evidências batem com o texto")
print(f"domínio: {len(final.edges) - len(fora_do_dominio)}/{len(final.edges)} arestas dentro do esquema")
assert not desalinhadas, desalinhadas
assert not fora_do_dominio, fora_do_dominio

print(f"\nNós ({len(nos)}):", nos["type"].value_counts().to_dict())
mostrar(nos[["node_id", "type", "label", "attributes"]], n=60, largura=80)

offsets: 38/38 evidências batem com o texto
domínio: 54/54 arestas dentro do esquema

Nós (45): {'Concept': 14, 'Finding': 9, 'Exam': 6, 'Symptom': 3, 'Outcome': 3, 'AnatomicalSite': 3, 'Diagnosis': 2, 'Treatment': 2, 'ExamResult': 2, 'Patient': 1}
node_id           type                                     label                                                                       attributes
     P1        Patient                        case PMC5137649_01                     article_id=PMC5137649; age=44; age_unit=years; gender=Female
     S1        Symptom right flank lower quadrant abdominal pain                                                polarity=present; duration=3 days
     S2        Symptom                                    nausea                                                                 polarity=present
     S3        Symptom                              constipation                                                                 polarity=present
     E1           Exa

In [13]:
rotulo_de = {n.node_id: rotulo_final(n) for n in final.nodes}
vista = pd.DataFrame(
    [
        {
            "edge_id": e.edge_id,
            "origem": f"{e.source_id} {curto(rotulo_de[e.source_id], 28)}",
            "relation": e.relation,
            "destino": f"{e.target_id} {curto(rotulo_de[e.target_id], 28)}",
            "trigger / estratégia": e.attributes.get("trigger", e.attributes.get("strategy")),
            "evidência": curto(e.attributes.get("evidence_text", ""), 50),
        }
        for e in final.edges
    ]
)
print(f"Arestas ({len(arestas)}):", arestas["relation"].value_counts().to_dict())
mostrar(vista, n=80, largura=60)

Arestas (54): {'SAME_AS': 16, 'LOCATED_IN': 8, 'UNDERWENT_EXAM': 6, 'HAS_FINDING': 6, 'HAS_SYMPTOM': 3, 'HAS_OUTCOME': 3, 'REVEALS': 3, 'DIAGNOSED_WITH': 2, 'HAS_RESULT': 2, 'TREATED_WITH': 2, 'SUPPORTS': 2, 'REVISES': 1}
edge_id                          origem       relation                          destino               trigger / estratégia                                          evidência
     e1           P1 case PMC5137649_01    HAS_SYMPTOM  S1 right flank lower quadrant …                     presented with presented with a 3-day history of right flank and…
     e2           P1 case PMC5137649_01    HAS_SYMPTOM                        S2 nausea                     presented with presented with a 3-day history of right flank and…
     e3           P1 case PMC5137649_01    HAS_SYMPTOM                  S3 constipation                     presented with presented with a 3-day history of right flank and…
     e4           P1 case PMC5137649_01 DIAGNOSED_WITH  D1 mucinous pancreatic cys

Atributos expandidos, um bloco por tipo de entidade:

In [14]:
for tipo, grupo in Counter(n.type for n in final.nodes).items():
    linhas = [{"node_id": n.node_id, "label": curto(rotulo_final(n), 40), **n.attributes} for n in final.nodes if n.type == tipo]
    df_tipo = pd.DataFrame(linhas).dropna(axis=1, how="all")
    print(f"\n{tipo} ({grupo})")
    mostrar(df_tipo, n=12, largura=35)


Patient (1)
node_id              label article_id age age_unit gender
     P1 case PMC5137649_01 PMC5137649  44    years Female

Symptom (3)
node_id                               label polarity duration
     S1 right flank lower quadrant abdom...  present   3 days
     S2                              nausea  present      NaN
     S3                        constipation  present      NaN

Exam (6)
node_id                     label   modality abbreviation
     E1       computed tomography    imaging          NaN
     E2     endoscopic ultrasound  endoscopy          EUS
     E3    fine-needle aspiration  pathology          FNA
     E4                 endoscopy  endoscopy          NaN
     E5  carcinoembryonic antigen laboratory          NaN
     E6 carbohydrate antigen 19-9 laboratory          NaN

Finding (9)
node_id                     label    source polarity certainty               size
     F1                      cyst pathology  present confirmed           6 x 9 cm
     F2          

## 12. Exportação
As duas tabelas vão para `data/processed/<case_id>-nodes.csv` e `-edges.csv`.

In [15]:
SAIDA = ROOT / "data" / "processed"
SAIDA.mkdir(parents=True, exist_ok=True)
caminho_nos = SAIDA / f"{CASE_ID}-nodes.csv"
caminho_arestas = SAIDA / f"{CASE_ID}-edges.csv"
nos.to_csv(caminho_nos, index=False)
arestas.to_csv(caminho_arestas, index=False)
print(f"{caminho_nos.relative_to(ROOT)}: {len(nos)} linhas")
print(f"{caminho_arestas.relative_to(ROOT)}: {len(arestas)} linhas")

data/processed/PMC5137649_01-nodes.csv: 45 linhas
data/processed/PMC5137649_01-edges.csv: 54 linhas
